# DESCRIPTION (inspired by Leetcode.com)

Given a list of intervals intervals and an interval newInterval, write a function to insert newInterval into a list of existing, non-overlapping, and sorted intervals based on their starting points. The function should ensure that after the new interval is added, the list remains sorted without any overlapping intervals, merging them if needed.

Two intervals are considered overlapping if they share any common time, including if one ends exactly when another begins (e.g., [1,4] and [4,7] overlap and should be merged into [1,7]).

Input:

intervals = [[1,3],[6,9]]
newInterval = [2,5]

Output: [[1,5],[6,9]]

Explanation: The new interval [2,5] overlaps with [1,3], so they are merged into [1,5].

---

Input:

intervals = [[1,2],[3,5],[6,7],[8,10]]
newInterval = [5,6]

Output: [[1,2],[3,7],[8,10]]

Explanation: The new interval [5,6] touches [3,5] and [6,7], so all three are merged into [3,7].

In [ ]:
class my_Solution:
    def insertIntervals(self, intervals: list[list[int]], newInterval: list[int]) -> list[list[int]]:
        # this solution will mutate the intervals list which is what the problem description expects
        # append a copy of newInterval to the end of intervals and will update each interval as needed 
        # making a copy instead of using the original newInterval is to avoid mutating it.
        intervals.append([newInterval[0],newInterval[1]])

        # start traversing from the end, so that when I need to delete an interval from the end of the list, the index won't get messed up.
        for i in range(len(intervals)-1, 0, -1):
            if intervals[i-1][1] < intervals[i][0]:
                # no action needed, the newInterval is larger than the largest interval in the list    
                break
            elif intervals[i][1] < intervals[i-1][0]:
                # there is no overlap, but the interval at position i is smaller than the previous one, so swap their place              
                intervals[i-1][0], intervals[i][0] = intervals[i][0], intervals[i-1][0]
                intervals[i-1][1], intervals[i][1] = intervals[i][1], intervals[i-1][1]
            else:
                # there is overlap, merge interval[i] with interval[i-1], and delete interval[i]
                intervals[i-1][0] = min(intervals[i][0], intervals[i-1][0])
                intervals[i-1][1] = max(intervals[i][1], intervals[i-1][1])
                del intervals[i]
            
        return intervals

# FEEDBACK

Nice work — your solution is correct and passes all tests.

A couple of interview-level notes:

Your approach is valid, but it mutates the input list with append, swaps, and del. That’s fine if allowed, but in interviews it’s good to mention this upfront since some callers may not expect intervals to change.

The reverse traversal is clever, but it’s a bit harder to reason about than a forward merge. Be ready to explain why swapping adjacent intervals preserves correctness.

One subtle edge case: using newInterval.copy() avoids aliasing, which is good. But you still mutate newInterval indirectly only through the copy, so that behavior is safe.

Overall: solid solution, and you handled merging logic efficiently in O(n).

In [64]:
class Solution:
    def insertIntervals(self, intervals: list[list[int]], newInterval: list[int]) -> list[list[int]]:
        merge = []
        i = 0

        while i < len(intervals) and intervals[i][1] < newInterval[0]:
            merge.append(intervals[i])
            i += 1

        start, end = newInterval[0], newInterval[1]

        while i < len(intervals) and intervals[i][0] <= end:
            start = min(intervals[i][0], start)
            end = max(intervals[i][1], end)
            i += 1
        merge.append([start,end])

        for j in range(i, len(intervals)):
            merge.append(intervals[j])

        return merge


In [26]:
from dataclasses import dataclass
from typing import Callable

@dataclass(frozen=True)
class Input:
    intervals: list[list[int]]
    new_interval: list[int]

@dataclass(frozen=True)
class Test:
    input: Input
    expected_result: list[list[int]]

def run_tests(tests: list[Test], func: Callable[[list[list[int]], list[int]], list[list[int]]]):
    for test in tests:
        original_list = str(test.input.intervals)
        result = func(test.input.intervals, test.input.new_interval)
        if result == test.expected_result:
            print(f"Test passed for {original_list} <- {test.input.new_interval}")
        else:
            print(f"Test failed for {original_list} <- {test.input.new_interval}. Expected: {test.expected_result}, Actual: {result}")

In [65]:
tests = [
    Test(Input([[1,3],[6,9]],[2,5]), [[1,5],[6,9]]),
    Test(Input([[1,3],[6,9]],[1,3]), [[1,3],[6,9]]),
    Test(Input([[1,3],[6,9]],[2,2]), [[1,3],[6,9]]),
    Test(Input([[1,3],[6,9]],[7,8]), [[1,3],[6,9]]),
    Test(Input([[1,3],[6,9]],[0,1]), [[0,3],[6,9]]),
    Test(Input([[1,3],[6,9]],[0,10]), [[0,10]]),
    Test(Input([[2,3],[6,9]],[0,1]), [[0,1],[2,3],[6,9]]),
    Test(Input([[1,3],[6,9]],[10,12]), [[1,3],[6,9],[10,12]]),
    Test(Input([[1,2],[3,5],[6,7],[8,10]],[5,6]), [[1,2],[3,7],[8,10]])
]

run_tests(tests, Solution().insertIntervals)

Test passed for [[1, 3], [6, 9]] <- [2, 5]
Test passed for [[1, 3], [6, 9]] <- [1, 3]
Test passed for [[1, 3], [6, 9]] <- [2, 2]
Test passed for [[1, 3], [6, 9]] <- [7, 8]
Test passed for [[1, 3], [6, 9]] <- [0, 1]
Test passed for [[1, 3], [6, 9]] <- [0, 10]
Test passed for [[2, 3], [6, 9]] <- [0, 1]
Test passed for [[1, 3], [6, 9]] <- [10, 12]
Test passed for [[1, 2], [3, 5], [6, 7], [8, 10]] <- [5, 6]
